In [1]:
from tensorflow import keras
from keras.models import Model
from keras import layers
from keras.layers import Dense, Input, BatchNormalization, Activation
from keras.layers import Conv2D, SeparableConv2D, MaxPooling2D, GlobalAveragePooling2D, GlobalMaxPooling2D
#from keras.applications.imagenet_utils import _obtain_input_shape
#from keras.utils.data_utils import get_file
from sklearn.model_selection import KFold
from keras.datasets import mnist
from PIL import Image
# for creating a one hot vector for labels
#from keras.utils import np_utils
from IPython.display import display, Image
#import the models
from keras import Model
#add layers
from keras import layers
#add optimizer
from keras import optimizers
#add loss function 
from keras import losses
import random

import keras
import numpy as np
import pandas as pd 
import os
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import tensorflow as tf
import tensorflow.keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, Dropout, MaxPooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import json
from skimage.metrics import structural_similarity as ssim

print("Hello world.")

2026-05-29 13:42:45.760754: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780062166.233704      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780062166.401049      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780062167.451961      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780062167.452021      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780062167.452028      58 computation_placer.cc:177] computation placer alr

Hello world.


In [2]:
import numpy as np
import cv2

def apply_perturbations(images, perturbation):
    """
    Apply a single perturbation to normalized images.

    Parameters:
    - images: list or array of preprocessed images (H, W, C), normalized [0,1]
    - perturbation: dict specifying type and parameters
        Examples:
        {'type':'gaussian_blur','kernel_size':5}
        {'type':'noise','mean':0,'std':0.05}
        {'type':'brightness','factor':0.2}
        {'type':'hue_shift','shift':10}

    Returns:
    - perturbed_images: np.array of perturbed images
    """
    perturbed_images = []

    for img in images:
        img_copy = img.copy()
        p_type = perturbation.get('type')

        if p_type == 'gaussian_blur':
            k = perturbation.get('kernel_size', 5)
            if k % 2 == 0:
                k += 1
            img_copy = cv2.GaussianBlur(img_copy, (k, k), 0)

        elif p_type == 'noise':
            mean = perturbation.get('mean', 0)
            std = perturbation.get('std', 0.05)
            noise = np.random.normal(mean, std, img_copy.shape)
            img_copy = np.clip(img_copy + noise, 0, 1)

        elif p_type == 'brightness':
            factor = perturbation.get('factor', 0.2)
            img_copy = np.clip(img_copy + factor, 0, 1)

        elif p_type == 'hue_shift':
            hsv = cv2.cvtColor((img_copy * 255).astype(np.uint8), cv2.COLOR_RGB2HSV)
            hsv[..., 0] = (hsv[..., 0] + perturbation.get('shift', 10)) % 180
            img_copy = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB).astype("float32") / 255.0

        perturbed_images.append(img_copy)

    return np.array(perturbed_images, dtype=np.float32)




def create_mask_from_json(json_file, img_size):
    # If file does not exist or is empty, return blank mask
    if not os.path.exists(json_file) or os.path.getsize(json_file) == 0:
        print(f"Warning: JSON file missing or empty: {json_file}")
        return np.zeros((img_size, img_size), dtype=np.uint8)

    # read json
    with open(json_file, "r") as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            print(f"Warning: JSON file invalid: {json_file}")
            return np.zeros((img_size, img_size), dtype=np.uint8)

    mask = np.zeros((img_size, img_size), dtype=np.uint8)

    # loop over each shape (polygon)
    for shape in data.get("shapes", []):
        pts = np.array(shape["points"], dtype=np.int32)
        pts = pts.reshape((-1, 1, 2))

        # fill polygon
        cv2.fillPoly(mask, [pts], 255)

    return mask

In [3]:
x = []
label = []
x_validation = []
label_validation = []
x_test = []
image_test= []
label_test = []
mask_test = []
img_size = 224

test_range  = []

In [4]:
image_folder0 = "/kaggle/input/datasets/mahmoudshaheen1134/plant-leaf-image-dataset/New_data/Apple"
csv_file0 = pd.read_csv("/kaggle/input/datasets/ragibmehedi/egypli-split/EGYPLI_split/apple_split.csv")
json_folder = "/kaggle/input/datasets/ragibmehedi/egypli-annotated-json-files/annotated/Apple"


columns = csv_file0.columns
print(list(columns))

for i in range(0, len(csv_file0)):
    image_name = csv_file0.loc[i]["image"]
    json_name = image_name.replace(".jpg", ".json")
    fold = csv_file0.loc[i]["used_for"]
    
    image = cv2.imread(image_folder0 +'/'+ image_name)
    image_original = image
    #image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    image = cv2.resize(image, (img_size, img_size))
    image = image.astype("float32") / 255.0

    json_file = json_folder + '/' +json_name

    if fold == "testing":
        x_test.append(image)
        label_test.append([0])
        mask = create_mask_from_json( json_file , image_original.shape[0])
        #mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        mask = cv2.resize(mask, (img_size, img_size))
        mask_test.append(mask)
    elif fold == "validation":
        x_validation.append(image)
        label_validation.append([0])
    else:
        x.append(image)
        label.append([0])


test_range.append(len(x_test))
print("Hello world.")

['Unnamed: 0', 'image', 'used_for']
Hello world.


In [5]:
image_folder1 = "/kaggle/input/datasets/mahmoudshaheen1134/plant-leaf-image-dataset/New_data/Berry"
csv_file1 = pd.read_csv("/kaggle/input/datasets/ragibmehedi/egypli-split/EGYPLI_split/berry_split.csv")
json_folder = "/kaggle/input/datasets/ragibmehedi/egypli-annotated-json-files/annotated/Berry"


columns = csv_file1.columns
print(list(columns))

for i in range(0, len(csv_file1)):
    image_name = csv_file1.loc[i]["image"]
    json_name = image_name.replace(".jpg", ".json")
    fold = csv_file1.loc[i]["used_for"]
    
    image = cv2.imread(image_folder1 +'/'+ image_name)
    image_original = image
    #image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    image = cv2.resize(image, (img_size, img_size))
    image = image.astype("float32") / 255.0

    json_file = json_folder + '/' +json_name

    if fold == "testing":
        x_test.append(image)
        label_test.append([1])
        mask = create_mask_from_json( json_file , image_original.shape[0])
        #mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        mask = cv2.resize(mask, (img_size, img_size))
        mask_test.append(mask)
    elif fold == "validation":
        x_validation.append(image)
        label_validation.append([1])
    else:
        x.append(image)
        label.append([1])


test_range.append(len(x_test))
print("Hello world.")

['Unnamed: 0', 'image', 'used_for']
Hello world.


In [6]:
image_folder2 = "/kaggle/input/datasets/mahmoudshaheen1134/plant-leaf-image-dataset/New_data/Fig"
csv_file2 = pd.read_csv("/kaggle/input/datasets/ragibmehedi/egypli-split/EGYPLI_split/fig_split.csv")
json_folder = "/kaggle/input/datasets/ragibmehedi/egypli-annotated-json-files/annotated/Fig"


columns = csv_file2.columns
print(list(columns))

for i in range(0, len(csv_file2)):
    image_name = csv_file2.loc[i]["image"]
    json_name = image_name.replace(".jpg", ".json")
    fold = csv_file2.loc[i]["used_for"]
    
    image = cv2.imread(image_folder2 +'/'+ image_name)
    image_original = image
    #image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    image = cv2.resize(image, (img_size, img_size))
    image = image.astype("float32") / 255.0

    json_file = json_folder + '/' +json_name

    if fold == "testing":
        x_test.append(image)
        label_test.append([2])
        mask = create_mask_from_json( json_file , image_original.shape[0])
        #mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        mask = cv2.resize(mask, (img_size, img_size))
        mask_test.append(mask)
    elif fold == "validation":
        x_validation.append(image)
        label_validation.append([2])
    else:
        x.append(image)
        label.append([2])


test_range.append(len(x_test))
print("Hello world.")

['Unnamed: 0', 'image', 'used_for']
Hello world.


In [7]:
image_folder3 = "/kaggle/input/datasets/mahmoudshaheen1134/plant-leaf-image-dataset/New_data/Guava"
csv_file3 = pd.read_csv("/kaggle/input/datasets/ragibmehedi/egypli-split/EGYPLI_split/guava_split.csv")
json_folder = "/kaggle/input/datasets/ragibmehedi/egypli-annotated-json-files/annotated/Guava"


columns = csv_file3.columns
print(list(columns))

for i in range(0, len(csv_file3)):
    image_name = csv_file3.loc[i]["image"]
    json_name = image_name.replace(".jpg", ".json")
    fold = csv_file3.loc[i]["used_for"]
    
    image = cv2.imread(image_folder3 +'/'+ image_name)
    image_original = image
    #image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    image = cv2.resize(image, (img_size, img_size))
    image = image.astype("float32") / 255.0

    json_file = json_folder + '/' +json_name

    if fold == "testing":
        x_test.append(image)
        label_test.append([3])
        mask = create_mask_from_json( json_file , image_original.shape[0])
        #mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        mask = cv2.resize(mask, (img_size, img_size))
        mask_test.append(mask)
    elif fold == "validation":
        x_validation.append(image)
        label_validation.append([3])
    else:
        x.append(image)
        label.append([3])

test_range.append(len(x_test))

print("Hello world.")

['Unnamed: 0', 'image', 'used_for']
Hello world.


In [8]:
image_folder4 = "/kaggle/input/datasets/mahmoudshaheen1134/plant-leaf-image-dataset/New_data/Orange"
csv_file4 = pd.read_csv("/kaggle/input/datasets/ragibmehedi/egypli-split/EGYPLI_split/orange_split.csv")
json_folder = "/kaggle/input/datasets/ragibmehedi/egypli-annotated-json-files/annotated/Orange"


columns = csv_file4.columns
print(list(columns))

for i in range(0, len(csv_file4)):
    image_name = csv_file4.loc[i]["image"]
    json_name = image_name.replace(".jpg", ".json")
    fold = csv_file4.loc[i]["used_for"]
    
    image = cv2.imread(image_folder4 +'/'+ image_name)
    image_original = image
    #image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    image = cv2.resize(image, (img_size, img_size))
    image = image.astype("float32") / 255.0

    json_file = json_folder + '/' +json_name

    if fold == "testing":
        x_test.append(image)
        label_test.append([4])
        mask = create_mask_from_json( json_file , image_original.shape[0])
        #mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        mask = cv2.resize(mask, (img_size, img_size))
        mask_test.append(mask)
    elif fold == "validation":
        x_validation.append(image)
        label_validation.append([4])
    else:
        x.append(image)
        label.append([4])


test_range.append(len(x_test))
print("Hello world.")

['Unnamed: 0', 'image', 'used_for']
Hello world.


In [9]:
image_folder5 = "/kaggle/input/datasets/mahmoudshaheen1134/plant-leaf-image-dataset/New_data/Palm"
csv_file5 = pd.read_csv("/kaggle/input/datasets/ragibmehedi/egypli-split/EGYPLI_split/palm_split.csv")
json_folder = "/kaggle/input/datasets/ragibmehedi/egypli-annotated-json-files/annotated/Palm"


columns = csv_file5.columns
print(list(columns))

for i in range(0, len(csv_file5)):
    image_name = csv_file5.loc[i]["image"]
    json_name = image_name.replace(".jpg", ".json")
    fold = csv_file5.loc[i]["used_for"]
    
    image = cv2.imread(image_folder5 +'/'+ image_name)
    image_original = image
    #image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    image = cv2.resize(image, (img_size, img_size))
    image = image.astype("float32") / 255.0

    json_file = json_folder + '/' +json_name

    if fold == "testing":
        x_test.append(image)
        label_test.append([5])
        mask = create_mask_from_json( json_file , image_original.shape[0])
        #mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        mask = cv2.resize(mask, (img_size, img_size))
        mask_test.append(mask)
    elif fold == "validation":
        x_validation.append(image)
        label_validation.append([5])
    else:
        x.append(image)
        label.append([5])

test_range.append(len(x_test))
print("Hello world.")

['Unnamed: 0', 'image', 'used_for']
Hello world.


In [10]:
image_folder6 = "/kaggle/input/datasets/mahmoudshaheen1134/plant-leaf-image-dataset/New_data/Persimmon"
csv_file6 = pd.read_csv("/kaggle/input/datasets/ragibmehedi/egypli-split/EGYPLI_split/persimmon_split.csv")
json_folder = "/kaggle/input/datasets/ragibmehedi/egypli-annotated-json-files/annotated/Persimmon"


columns = csv_file6.columns
print(list(columns))

for i in range(0, len(csv_file6)):
    image_name = csv_file6.loc[i]["image"]
    json_name = image_name.replace(".jpg", ".json")
    fold = csv_file6.loc[i]["used_for"]
    
    image = cv2.imread(image_folder6 +'/'+ image_name)
    image_original = image
    #image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    image = cv2.resize(image, (img_size, img_size))
    image = image.astype("float32") / 255.0

    json_file = json_folder + '/' +json_name

    if fold == "testing":
        x_test.append(image)
        label_test.append([6])
        mask = create_mask_from_json( json_file , image_original.shape[0])
        #mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        mask = cv2.resize(mask, (img_size, img_size))
        mask_test.append(mask)
    elif fold == "validation":
        x_validation.append(image)
        label_validation.append([6])
    else:
        x.append(image)
        label.append([6])


test_range.append(len(x_test))
print("Hello world.")

['Unnamed: 0', 'image', 'used_for']
Hello world.


In [11]:
image_folder7 = "/kaggle/input/datasets/mahmoudshaheen1134/plant-leaf-image-dataset/New_data/Tomato"
csv_file7 = pd.read_csv("/kaggle/input/datasets/ragibmehedi/egypli-split/EGYPLI_split/tomato_split.csv")
json_folder = "/kaggle/input/datasets/ragibmehedi/egypli-annotated-json-files/annotated/Tomato"


columns = csv_file7.columns
print(list(columns))

for i in range(0, len(csv_file7)):
    image_name = csv_file7.loc[i]["image"]
    json_name = image_name.replace(".jpg", ".json")
    fold = csv_file7.loc[i]["used_for"]
    
    image = cv2.imread(image_folder7 +'/'+ image_name)
    image_original = image
    #image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    image = cv2.resize(image, (img_size, img_size))
    image = image.astype("float32") / 255.0

    json_file = json_folder + '/' +json_name

    if fold == "testing":
        x_test.append(image)
        label_test.append([7])
        mask = create_mask_from_json( json_file , image_original.shape[0])
        #mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        mask = cv2.resize(mask, (img_size, img_size))
        mask_test.append(mask)
    elif fold == "validation":
        x_validation.append(image)
        label_validation.append([7])
    else:
        x.append(image)
        label.append([7])


test_range.append(len(x_test))
print("Hello world.")

['Unnamed: 0', 'image', 'used_for']
Hello world.


In [12]:
x = np.asarray(x)
x_validation = np.asarray(x_validation)
x_test = np.asarray(x_test)

print(len(x)," ", len(x_validation)," ", len(x_test))

2868   359   361


In [13]:
from tensorflow.keras.utils import to_categorical 

label = np.array(label)
label = to_categorical(label , 8 )
print(len(label))



label_validation = np.array(label_validation)
label_validation = to_categorical(label_validation , 8 )
print(len(label_validation))

label_test = np.array(label_test)
label_test = to_categorical(label_test , 8 )
print(len(label_test))

2868
359
361


In [14]:

from tensorflow.keras.applications import InceptionResNetV2
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Input layer
inputs = Input(shape=(224, 224, 3))

# Load InceptionResNetV2 backbone
base_model = InceptionResNetV2(
    weights="imagenet",
    include_top=False,
    input_tensor=inputs
)

# Freeze backbone
base_model.trainable = False

# Classification head
y = base_model.output
y = GlobalAveragePooling2D()(y)
y = Dropout(0.3)(y)
y = Dense(128, activation="relu")(y)
outputs = Dense(8, activation="softmax")(y)

# Final model
model = Model(inputs=inputs, outputs=outputs)

# Compile
model.compile(
    optimizer=Adam(learning_rate=5e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()




I0000 00:00:1780062228.969332      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1780062228.975575      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


219055592/219055592 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 111, 111,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 111, 111,  │         96 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 111, 111,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 109, 109,  │      9,216 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 109, 109,  │         96 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 109, 109,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 109, 109,  │     18,432 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 109, 109,  │        192 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 109, 109,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 54, 54,    │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 54, 54,    │      5,120 │ max_pooling2d[0]… │
│                     │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 54, 54,    │        240 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 54, 54,    │          0 │ batch_normalizat… │
│ (Activation)        │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 52, 52,    │    138,240 │ activation_3[0][… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 52, 52,    │        576 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 52, 52,    │          0 │ batch_normalizat

 Total params: 54,534,504 (208.03 MB)

 Trainable params: 197,768 (772.53 KB)

 Non-trainable params: 54,336,736 (207.28 MB)

In [15]:
checkpoint1 = tf.keras.callbacks.ModelCheckpoint(
    "InceptionResNetV2_EGYPLI.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

history1 = model.fit(
    x, label,
    validation_data=(x_validation, label_validation),
    epochs= 100, 
    batch_size=16,
    callbacks=[checkpoint1]
)

Epoch 1/100


I0000 00:00:1780062256.775259     128 service.cc:152] XLA service 0x7ad07c004dd0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780062256.775330     128 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1780062256.775334     128 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1780062262.561073     128 cuda_dnn.cc:529] Loaded cuDNN version 91002


  1/180 ━━━━━━━━━━━━━━━━━━━━ 1:29:57 30s/step - accuracy: 0.1250 - loss: 2.5684

I0000 00:00:1780062270.729047     128 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - accuracy: 0.3186 - loss: 1.8938
Epoch 1: val_accuracy improved from None to 0.75487, saving model to InceptionResNetV2_EGYPLI.keras

Epoch 1: finished saving model to InceptionResNetV2_EGYPLI.keras
180/180 ━━━━━━━━━━━━━━━━━━━━ 78s 267ms/step - accuracy: 0.4585 - loss: 1.5322 - val_accuracy: 0.7549 - val_loss: 0.8866
Epoch 2/100
179/180 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.7045 - loss: 0.9305
Epoch 2: val_accuracy improved from 0.75487 to 0.82451, saving model to InceptionResNetV2_EGYPLI.keras

Epoch 2: finished saving model to InceptionResNetV2_EGYPLI.keras
180/180 ━━━━━━━━━━━━━━━━━━━━ 18s 97ms/step - accuracy: 0.7423 - loss: 0.8264 - val_accuracy: 0.8245 - val_loss: 0.5764
Epoch 3/100
179/180 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.8353 - loss: 0.5986
Epoch 3: val_accuracy improved from 0.82451 to 0.87187, saving model to InceptionResNetV2_EGYPLI.keras

Epoch 3: finished saving model to InceptionResNetV2_EGYPLI.keras
180/

In [16]:
test_loss, test_acc = model.evaluate(x_test, label_test)
print("Test Accuracy:", test_acc)

12/12 ━━━━━━━━━━━━━━━━━━━━ 19s 828ms/step - accuracy: 0.9917 - loss: 0.0224
Test Accuracy: 0.9916897416114807


In [17]:
from tensorflow.keras.models import load_model

model_path = "/kaggle/working/InceptionResNetV2_EGYPLI.keras"
model = load_model(model_path)
print("Model loaded successfully!")

Model loaded successfully!


In [18]:
test_loss, test_accuracy = model.evaluate(x_test, label_test )
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

12/12 ━━━━━━━━━━━━━━━━━━━━ 23s 628ms/step - accuracy: 0.9889 - loss: 0.0280
Test Accuracy: 0.9889
Test Loss: 0.0280


In [19]:
predictions = model.predict(x_test)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(label_test, axis=1)

12/12 ━━━━━━━━━━━━━━━━━━━━ 23s 1s/step 


In [20]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

print("\nClassification Report:")
print(classification_report(true_classes, predicted_classes))

# Generate a confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(true_classes, predicted_classes))


Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        52
           1       1.00      0.97      0.99        34
           2       0.98      1.00      0.99        52
           3       0.98      0.98      0.98        52
           4       0.98      1.00      0.99        55
           5       0.98      0.96      0.97        47
           6       1.00      1.00      1.00        53
           7       1.00      1.00      1.00        16

    accuracy                           0.99       361
   macro avg       0.99      0.99      0.99       361
weighted avg       0.99      0.99      0.99       361


Confusion Matrix:
[[52  0  0  0  0  0  0  0]
 [ 0 33  1  0  0  0  0  0]
 [ 0  0 52  0  0  0  0  0]
 [ 0  0  0 51  0  1  0  0]
 [ 0  0  0  0 55  0  0  0]
 [ 0  0  0  1  1 45  0  0]
 [ 0  0  0  0  0  0 53  0]
 [ 0  0  0  0  0  0  0 16]]


In [22]:
while(True):
     a = 5

KeyboardInterrupt: 